# Colab: first real-data PINN runs (M6/M7)

**What this batch does.** It trains the physics-informed model on the real CGWB wells for the first time:
- one model for the whole region (the *regional PINN*, the main comparison), with 4 settings of how much to trust the readings;
- one model per zone (the *zoned PINN*, our method), 4 zones;
- one synthetic check: can about 30 frequently-read wells show the seasonal pumping pattern? (for the telemetry idea)

Results are scored on the **validation** wells only. The test wells are not looked at yet.

**Before you start:**
1. Wait until the re-check batch (`colab_m2_recheck`) has finished and you have downloaded `m2_v2_results.zip`.
2. Upload the new `pinn_code.zip` (from the laptop's `pinn` folder) to `MyDrive/pinn/`, replacing the old one.
3. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** About 3.5 hours (5 regional runs ~25 min each, 4 zone runs ~15 min each, 1 synthetic run). Keep the tab open.

**If Colab disconnects:** just do **Run all** again. Runs that already finished are skipped.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs (most important first)

| Run | Question it answers |
|---|---|
| regional, default (data weight 10, well corrections 100) | the main comparison model |
| zones 1–4, default | our method, same settings |
| regional, data weight 1 / 100 | how much should the model trust noisy readings? |
| regional, well corrections weaker (1) / stronger (1000) | how much of each well's level may be put down to elevation error? |
| synthetic d3 | can ~30 frequently read wells resolve seasonal pumping? |

**How the setting is chosen (fixed before the runs):** lowest combined validation error on tasks A and B, each relative to the best baseline for that task.

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real1.log"
RUNS = [
    ("*_real_L0_regional_s0_base", "--regional --tag base"),
    *[(f"*_real_L0_kmeansk5_z{z}_s0_base", f"--zone {z} --tag base") for z in (1, 2, 3, 4)],
    ("*_real_L0_regional_s0_w1", "--regional --tag w1 --set training.weighting.init=1"),
    ("*_real_L0_regional_s0_w100", "--regional --tag w100 --set training.weighting.init=100"),
    ("*_real_L0_regional_s0_off1", "--regional --tag off1 --set training.well_offsets.weight=1"),
    ("*_real_L0_regional_s0_off1000", "--regional --tag off1000 --set training.well_offsets.weight=1000"),
]
for pattern, args in RUNS:
    if done(pattern):
        print("already done:", pattern)
        continue
    !python -u scripts/run_real.py {args} --stage-only val 2>&1 | tee -a {LOG}

In [ ]:
!python scripts/make_synthetic.py
if not done("*_m2_d3_diag"):
    !python -u scripts/run_m2.py d3 --tag diag 2>&1 | tee -a {LOG}

## Bundle the results for Claude
This makes `MyDrive/pinn/real1_results.zip`. Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real1_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{DRIVE_ROOT}/runs/**/*", recursive=True):
        rel = os.path.relpath(path, f"{DRIVE_ROOT}/runs")
        if os.path.isfile(path) and ("_real_" in rel or "_m2_d3_" in rel or rel.endswith("real1.log")):
            z.write(path, rel)
print("wrote", out, f"({os.path.getsize(out) / 1e6:.2f} MB)")